# Notebook 09 — vLLM Quickstart and API Serving

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/09_vllm_quickstart_and_api_serving.ipynb)

Module 3 begins with the runtime most teams reach for when a plain Python baseline is no longer enough. This notebook shows how to think about vLLM setup, OpenAI-compatible serving, and first-pass benchmarking without assuming you have a large GPU host inside the notebook.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What you will build**
- Understand **Practical stance**
- Understand **Step 1: Add notebook helpers and artifact paths**
- Understand **Step 2: Define realistic vLLM deployment profiles**
- Understand **Step 3: Estimate weight and KV-cache budgets**


## What you will build

- a practical vLLM deployment profile sheet
- a launch command for `vllm serve` with 2026-friendly defaults
- OpenAI-compatible request payloads and streaming response shapes
- a notebook-local serving stub for smoke testing contracts
- a lightweight benchmark that compares a raw baseline against a vLLM-style serving profile

## Practical stance

We keep the control plane real and the heavy runtime optional. That means configuration, request contracts, and benchmark logic are concrete, while large-model execution is represented with simulations so the notebook remains usable on ordinary hardware.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

## Step 1: Add notebook helpers and artifact paths

We will export a launch manifest, request payload samples, and a benchmark report so the notebook leaves behind reusable serving artifacts.

In [ ]:
random.seed(9)

ARTIFACT_DIR = ARTIFACT_ROOT / "09_vllm_quickstart_and_api_serving"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

def display_frame(frame, columns=None):
    if columns is not None:
        frame = frame[columns]
    print(frame.to_markdown(index=False))

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define realistic vLLM deployment profiles

The model choice, context window, precision, and tensor parallel setting decide whether a deployment is small and cheap or large and throughput-oriented.

In [ ]:
deployment_profiles = [
    {"profile": "single-gpu-chat", "model": "Qwen3-8B-Instruct", "params_b": 8, "precision": "bf16", "precision_bytes": 2, "tensor_parallel_size": 1, "pipeline_parallel_size": 1, "max_model_len": 32768, "max_num_seqs": 16, "hidden_size": 4096, "layers": 32},
    {"profile": "dual-gpu-tooling", "model": "Mistral-Small-3.1-Instruct", "params_b": 24, "precision": "bf16", "precision_bytes": 2, "tensor_parallel_size": 2, "pipeline_parallel_size": 1, "max_model_len": 65536, "max_num_seqs": 24, "hidden_size": 5120, "layers": 40},
    {"profile": "throughput-cluster", "model": "Llama-3.3-70B-Instruct", "params_b": 70, "precision": "fp8", "precision_bytes": 1, "tensor_parallel_size": 8, "pipeline_parallel_size": 1, "max_model_len": 131072, "max_num_seqs": 40, "hidden_size": 8192, "layers": 80},
]

deployment_df = pd.DataFrame(deployment_profiles)
display_frame(deployment_df, ["profile", "model", "precision", "tensor_parallel_size", "max_model_len", "max_num_seqs"])

## Step 3: Estimate weight and KV-cache budgets

A deployment profile is only credible if its memory budget closes. The estimate below is rough on purpose, but it is good enough to catch obviously impossible launches.

In [ ]:
def estimate_vram_budget(row, kv_bytes_per_value=2):
    weight_gib = row["params_b"] * 1_000_000_000 * row["precision_bytes"] / (1024 ** 3)
    kv_gib = (
        row["max_num_seqs"]
        * row["max_model_len"]
        * row["hidden_size"]
        * row["layers"]
        * 2
        * kv_bytes_per_value
        / (1024 ** 3)
    )
    runtime_overhead_gib = max(3.0, weight_gib * 0.08)
    total_gib = weight_gib + kv_gib + runtime_overhead_gib
    return pd.Series({
        "weight_gib_est": round(weight_gib, 1),
        "kv_gib_est": round(kv_gib, 1),
        "runtime_overhead_gib_est": round(runtime_overhead_gib, 1),
        "total_gib_est": round(total_gib, 1),
    })

budget_df = pd.concat([deployment_df, deployment_df.apply(estimate_vram_budget, axis=1)], axis=1)
display_frame(budget_df, ["profile", "model", "weight_gib_est", "kv_gib_est", "runtime_overhead_gib_est", "total_gib_est"])

## Step 4: Choose a first serving target

For a quickstart, a single-GPU chat profile is the safest first path. Once the contract is stable, larger tensor-parallel targets become operational work instead of guesswork.

In [ ]:
selected_profile = budget_df.loc[budget_df["profile"] == "single-gpu-chat"].iloc[0].to_dict()
selected_profile

## Step 5: Build a 2026-style `vllm serve` command

The exact flag set changes over time, but the main ideas stay stable: explicit served model name, bounded concurrency, context limit, memory utilization target, and an OpenAI-compatible surface.

In [ ]:
launch_config = {
    "model": selected_profile["model"],
    "served_model_name": "castalia-mentor-chat",
    "host": "0.0.0.0",
    "port": 8000,
    "dtype": selected_profile["precision"],
    "tensor_parallel_size": int(selected_profile["tensor_parallel_size"]),
    "max_model_len": int(selected_profile["max_model_len"]),
    "max_num_seqs": int(selected_profile["max_num_seqs"]),
    "gpu_memory_utilization": 0.9,
    "api_key": "local-dev-key",
    "enable_prefix_caching": True,
    "disable_log_requests": False,
}

launch_flags = [
    "--served-model-name " + launch_config["served_model_name"],
    "--host " + launch_config["host"],
    "--port " + str(launch_config["port"]),
    "--dtype " + launch_config["dtype"],
    "--tensor-parallel-size " + str(launch_config["tensor_parallel_size"]),
    "--max-model-len " + str(launch_config["max_model_len"]),
    "--max-num-seqs " + str(launch_config["max_num_seqs"]),
    "--gpu-memory-utilization " + str(launch_config["gpu_memory_utilization"]),
    "--api-key " + launch_config["api_key"],
    "--enable-prefix-caching" if launch_config["enable_prefix_caching"] else "",
]
launch_flags = [flag for flag in launch_flags if flag]
launch_command = " ".join(["vllm", "serve", launch_config["model"], *launch_flags])
print(launch_command)

## Step 6: Record the client-side contract

Serving only becomes useful when the client settings are equally explicit. We keep the base URL, headers, and timeout policy together so later notebooks can reuse the same contract.

In [ ]:
client_config = {
    "base_url": "http://127.0.0.1:8000/v1",
    "api_key": launch_config["api_key"],
    "timeout_seconds": 45,
    "default_model": launch_config["served_model_name"],
}

client_headers = {
    "Authorization": "Bearer " + client_config["api_key"],
    "Content-Type": "application/json",
}

print(json.dumps({"client_config": client_config, "headers": client_headers}, indent=2))

## Step 7: Build OpenAI-compatible chat payloads

The safest baseline for broad ecosystem compatibility is still a chat-style request. Later notebooks add structured outputs and tool contracts on top of this foundation.

In [ ]:
chat_payload = {
    "model": client_config["default_model"],
    "messages": [
        {"role": "system", "content": "You are Castalia Mentor. Answer clearly and keep systems explanations concrete."},
        {"role": "user", "content": "Explain why vLLM keeps throughput high under bursty traffic."},
    ],
    "temperature": 0.2,
    "max_tokens": 180,
    "stream": False,
}

stream_payload = {**chat_payload, "stream": True, "max_tokens": 120}
completion_payload = {
    "model": client_config["default_model"],
    "prompt": "Summarize the role of the KV cache in one paragraph.",
    "max_tokens": 120,
    "temperature": 0.0,
}

print(json.dumps({"chat_payload": chat_payload, "stream_payload": stream_payload, "completion_payload": completion_payload}, indent=2))

## Step 8: Build a notebook-local serving stub

This stub does not run a model. It only validates the request and returns a response object that matches the OpenAI-compatible shape closely enough for smoke tests.

In [ ]:
def fake_vllm_chat(payload):
    prompt_text = " ".join(message["content"] for message in payload["messages"] if message["role"] != "system")
    content = (
        "vLLM keeps throughput high by scheduling tokens across many active sequences, "
        "reusing KV-cache state, and reducing idle GPU time that would appear in a plain sequential server. "
        "The exact win depends on prompt length, output length, and memory pressure."
    )
    prompt_tokens = max(48, int(len(prompt_text.split()) * 1.6))
    completion_tokens = min(payload.get("max_tokens", 120), 92)
    return {
        "id": "chatcmpl-sim-0009",
        "object": "chat.completion",
        "created": 1760000009,
        "model": payload["model"],
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": content},
                "finish_reason": "stop",
            }
        ],
        "usage": {
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": prompt_tokens + completion_tokens,
        },
    }

stub_response = fake_vllm_chat(chat_payload)
print(json.dumps(stub_response, indent=2))

## Step 9: Simulate a streaming response

Streaming matters because production clients often care about time-to-first-token more than total latency. We represent the stream as incremental chunk objects.

In [ ]:
def fake_stream_chunks(payload, chunk_size=18):
    full_text = fake_vllm_chat(payload)["choices"][0]["message"]["content"]
    chunks = []
    for index, start in enumerate(range(0, len(full_text), chunk_size)):
        delta = full_text[start:start + chunk_size]
        chunks.append({
            "id": "chatcmpl-sim-0009",
            "object": "chat.completion.chunk",
            "created": 1760000009 + index,
            "model": payload["model"],
            "choices": [{"index": 0, "delta": {"content": delta}, "finish_reason": None}],
        })
    chunks.append({
        "id": "chatcmpl-sim-0009",
        "object": "chat.completion.chunk",
        "created": 1760000019,
        "model": payload["model"],
        "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}],
    })
    return chunks

stream_chunks = fake_stream_chunks(stream_payload)
print(json.dumps(stream_chunks[:3], indent=2))
print("Chunk count:", len(stream_chunks))

## Step 10: Create a benchmark workload

We keep the workload mixed because serving systems do not receive one neat prompt size forever. The spread in prompt and output lengths is what makes scheduling hard.

In [ ]:
benchmark_requests = [
    {"request_id": "req-01", "arrival_ms": 0, "prompt_tokens": 420, "output_tokens": 160, "route": "chat"},
    {"request_id": "req-02", "arrival_ms": 8, "prompt_tokens": 680, "output_tokens": 220, "route": "chat"},
    {"request_id": "req-03", "arrival_ms": 15, "prompt_tokens": 250, "output_tokens": 90, "route": "extract"},
    {"request_id": "req-04", "arrival_ms": 24, "prompt_tokens": 910, "output_tokens": 180, "route": "chat"},
    {"request_id": "req-05", "arrival_ms": 32, "prompt_tokens": 300, "output_tokens": 60, "route": "classify"},
    {"request_id": "req-06", "arrival_ms": 45, "prompt_tokens": 1200, "output_tokens": 240, "route": "chat"},
    {"request_id": "req-07", "arrival_ms": 49, "prompt_tokens": 560, "output_tokens": 110, "route": "extract"},
    {"request_id": "req-08", "arrival_ms": 57, "prompt_tokens": 350, "output_tokens": 70, "route": "classify"},
]

workload_df = pd.DataFrame(benchmark_requests)
display_frame(workload_df)

## Step 11: Simulate a plain sequential server

The plain baseline is useful because it makes queue growth visible immediately. Even a good model can feel slow if requests only move one at a time.

In [ ]:
def simulate_sequential(records, prefill_tps=2400, decode_tps=30, overhead_ms=34):
    rows = []
    clock_ms = 0.0
    for record in records:
        queue_ms = max(0.0, clock_ms - record["arrival_ms"])
        prefill_ms = 1000 * record["prompt_tokens"] / prefill_tps
        decode_ms = 1000 * record["output_tokens"] / decode_tps
        latency_ms = queue_ms + prefill_ms + decode_ms + overhead_ms
        finish_ms = record["arrival_ms"] + latency_ms
        clock_ms = finish_ms
        rows.append({
            **record,
            "runtime": "raw_baseline",
            "queue_ms": round(queue_ms, 1),
            "prefill_ms": round(prefill_ms, 1),
            "decode_ms": round(decode_ms, 1),
            "latency_ms": round(latency_ms, 1),
            "throughput_tps": round(record["output_tokens"] / max(latency_ms / 1000, 0.001), 1),
        })
    return pd.DataFrame(rows)

baseline_df = simulate_sequential(workload_df.to_dict("records"))
display_frame(baseline_df, ["request_id", "queue_ms", "prefill_ms", "decode_ms", "latency_ms", "throughput_tps"])

## Step 12: Simulate a vLLM-style serving profile

This is still a simplified model, but it captures the main difference: requests arriving close together can share runtime work instead of waiting for the entire previous request to finish.

In [ ]:
def flush_dynamic_batch(batch, ready_ms, batch_id, prefill_tps=6800, decode_tps=78, overhead_ms=18):
    max_prompt = max(item["prompt_tokens"] for item in batch)
    max_output = max(item["output_tokens"] for item in batch)
    service_ms = 1000 * max_prompt / prefill_tps + 1000 * max_output / decode_tps + overhead_ms
    batch_finish = ready_ms + service_ms
    rows = []
    for item in batch:
        latency_ms = batch_finish - item["arrival_ms"]
        rows.append({
            **item,
            "runtime": "vllm_openai_server",
            "batch_id": batch_id,
            "dispatch_ms": round(ready_ms, 1),
            "latency_ms": round(latency_ms, 1),
            "throughput_tps": round(item["output_tokens"] / max(latency_ms / 1000, 0.001), 1),
        })
    return rows, batch_finish

def simulate_dynamic_vllm(records, max_batch_size=4, max_wait_ms=12):
    ordered = sorted(records, key=lambda item: item["arrival_ms"])
    rows = []
    batch = []
    batch_id = 0
    clock_ms = 0.0
    for record in ordered:
        if not batch:
            batch = [record]
            continue
        first_arrival = batch[0]["arrival_ms"]
        if len(batch) >= max_batch_size or record["arrival_ms"] - first_arrival > max_wait_ms:
            ready_ms = max(clock_ms, first_arrival + max_wait_ms)
            flushed_rows, clock_ms = flush_dynamic_batch(batch, ready_ms, batch_id)
            rows.extend(flushed_rows)
            batch_id += 1
            batch = [record]
        else:
            batch.append(record)
    if batch:
        first_arrival = batch[0]["arrival_ms"]
        ready_ms = max(clock_ms, first_arrival)
        flushed_rows, clock_ms = flush_dynamic_batch(batch, ready_ms, batch_id)
        rows.extend(flushed_rows)
    return pd.DataFrame(rows)

vllm_df = simulate_dynamic_vllm(workload_df.to_dict("records"))
display_frame(vllm_df, ["request_id", "batch_id", "dispatch_ms", "latency_ms", "throughput_tps"])

## Step 13: Compare the two serving profiles

The exact numbers are synthetic, but the shape should look familiar: less queue accumulation and higher delivered token rate when the server can batch intelligently.

In [ ]:
summary_rows = []
for frame in [baseline_df, vllm_df]:
    runtime_name = frame["runtime"].iloc[0]
    summary_rows.append({
        "runtime": runtime_name,
        "mean_latency_ms": round(frame["latency_ms"].mean(), 1),
        "p95_latency_ms": round(frame["latency_ms"].quantile(0.95), 1),
        "mean_throughput_tps": round(frame["throughput_tps"].mean(), 1),
    })

runtime_summary = pd.DataFrame(summary_rows)
display_frame(runtime_summary)

## Step 14: Visualize latency and throughput

A small chart is often enough to explain to teammates why a dedicated runtime is worth operating.

In [ ]:
ax = runtime_summary.plot(x="runtime", y=["mean_latency_ms", "p95_latency_ms"], kind="bar", rot=0, figsize=(8, 4), title="Latency comparison")
ax.set_ylabel("milliseconds")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "09_latency_comparison.png", dpi=160)
plt.show()

ax = runtime_summary.plot(x="runtime", y="mean_throughput_tps", kind="bar", rot=0, figsize=(7, 4), title="Mean delivered output tokens per second", color="#4c78a8")
ax.set_ylabel("tokens per second")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "09_throughput_comparison.png", dpi=160)
plt.show()

## Step 15: Prepare health and model probes

An OpenAI-compatible server still needs a small operational surface around it. Health checks and model listing are the first two probes almost every deployment needs.

In [ ]:
probe_examples = {
    "health": {"method": "GET", "url": "http://127.0.0.1:8000/health"},
    "models": {"method": "GET", "url": client_config["base_url"] + "/models"},
    "chat": {"method": "POST", "url": client_config["base_url"] + "/chat/completions"},
}

model_list_stub = {
    "object": "list",
    "data": [
        {"id": launch_config["served_model_name"], "object": "model", "owned_by": "open-source"},
    ],
}

print(json.dumps({"probe_examples": probe_examples, "model_list_stub": model_list_stub}, indent=2))

## Step 16: Render copy-paste smoke-test commands

The notebook should leave behind commands that can be used on a real Linux GPU host with only minor edits.

In [ ]:
smoke_test_commands = [
    "export OPENAI_BASE_URL=http://127.0.0.1:8000/v1",
    "export OPENAI_API_KEY=" + client_config["api_key"],
    launch_command,
    "curl -s http://127.0.0.1:8000/health",
    "curl -s http://127.0.0.1:8000/v1/models -H " + chr(34) + "Authorization: Bearer " + client_config["api_key"] + chr(34),
]

print("\n".join(smoke_test_commands))

## Step 17: Capture a Python client example

A plain `httpx` example is useful because it makes no assumptions about a higher-level SDK.

In [ ]:
python_client_example = "\n".join([
    "import httpx",
    "",
    "payload = " + json.dumps(chat_payload),
    "headers = " + json.dumps(client_headers),
    "response = httpx.post(" + json.dumps(client_config["base_url"] + "/chat/completions") + ", json=payload, headers=headers, timeout=45)",
    "print(response.json())",
])

print(python_client_example)

## Step 18: Export a serving manifest

A small manifest makes the notebook reusable by later modules and by anyone who needs to stand up the same service elsewhere.

In [ ]:
serving_manifest = {
    "notebook": "09_vllm_quickstart_and_api_serving",
    "launch_config": launch_config,
    "client_config": client_config,
    "budget_summary": runtime_summary.to_dict("records"),
    "selected_profile": selected_profile,
    "sample_payloads": {
        "chat_payload": chat_payload,
        "stream_payload": stream_payload,
        "completion_payload": completion_payload,
    },
    "artifacts": [
        "09_latency_comparison.png",
        "09_throughput_comparison.png",
    ],
}

write_json(ARTIFACT_DIR / "serving_manifest.json", serving_manifest)
print(json.dumps(serving_manifest, indent=2))

## Recap

You now have a practical vLLM quickstart: deployment profiles, a launch command, client contracts, OpenAI-compatible payloads, streaming response shapes, and a first benchmark that explains why optimized serving exists at all. The next notebook digs into the scheduler and the managed KV cache that make those gains possible.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** benchmark with a different model size and compare throughput. Document what changes and why.

**Exercise 2 — Build:** add a monitoring metric to the serving pipeline. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** simulate a failure scenario and verify the system recovers.

## 📝 Key Takeaways

- **What you will build** — revisit this section for reference
- **Practical stance** — revisit this section for reference
- **Step 1: Add notebook helpers and artifact paths** — revisit this section for reference
- **Step 2: Define realistic vLLM deployment profiles** — revisit this section for reference
- **Step 3: Estimate weight and KV-cache budgets** — revisit this section for reference


## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the systems/ directory